# Silo — Data Source Exploration

Quick experiments to confirm each API works, understand the response structure, and identify gaps before the build.

**Sources to test:**
1. USDA Market News API — IL soybean cash bids ← biggest unknown
2. yfinance — CBOT soybean futures (ZS=F)
3. NOAA — weather / drought signals
4. FRED — diesel prices + T-bill rate
5. Google Maps Distance Matrix — farm → buyer driving distance

**Demo scenario:** 42,000 bu soybeans, Springfield IL farm, two buyers.

In [ ]:
import requests
import httpx
import json
import pandas as pd
import numpy as np
import yfinance as yf
import os
import base64
from datetime import date, timedelta
from dotenv import load_dotenv
from pprint import pprint

# Load .env
load_dotenv()

# ── API keys from .env ────────────────────────────────────────────────────────
FRED_API_KEY = os.getenv("FRED_API_KEY", "")
GOOGLE_MAPS_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "")
USDA_API_KEY = os.getenv("USDA_API_KEY", "")
NOAA_CDO_TOKEN = os.getenv("NOAA_CDO_TOKEN", "")

TODAY = date.today().isoformat()
THIRTY_DAYS_AGO = (date.today() - timedelta(days=30)).isoformat()
print(f"Date range: {THIRTY_DAYS_AGO} → {TODAY}")
print(f"\nAPI Keys Loaded:")
print(f"  FRED_API_KEY: {'✓' if FRED_API_KEY else '✗'}")
print(f"  GOOGLE_MAPS_API_KEY: {'✓' if GOOGLE_MAPS_KEY else '✗'}")
print(f"  USDA_API_KEY: {'✓' if USDA_API_KEY else '✗'}")
print(f"  NOAA_CDO_TOKEN: {'✓' if NOAA_CDO_TOKEN else '✗'}")

# ── USDA auth method cache (determined by first successful call) ───────────────
_USDA_AUTH_METHOD = None

def usda_request(url, params=None, debug=False, **kwargs):
    """Make a request to USDA MARS API with proper authentication."""
    global _USDA_AUTH_METHOD
    
    if params is None:
        params = {}
    
    # If we've already found the working method, use it
    if _USDA_AUTH_METHOD:
        if debug:
            print(f"Using cached auth method: {_USDA_AUTH_METHOD}")
        
        if _USDA_AUTH_METHOD == "query":
            params["apiKey"] = USDA_API_KEY
        elif _USDA_AUTH_METHOD == "header_x_api_key":
            kwargs["headers"] = kwargs.get("headers", {})
            kwargs["headers"]["X-API-Key"] = USDA_API_KEY
        elif _USDA_AUTH_METHOD == "header_bearer":
            kwargs["headers"] = kwargs.get("headers", {})
            kwargs["headers"]["Authorization"] = f"Bearer {USDA_API_KEY}"
        elif _USDA_AUTH_METHOD == "basic":
            kwargs["auth"] = (USDA_API_KEY, "")
        
        return requests.get(url, params=params, **kwargs)
    
    # Try each method until one works
    methods = [
        ("query", lambda: requests.get(url, params={**params, "apiKey": USDA_API_KEY}, timeout=10)),
        ("header_x_api_key", lambda: requests.get(url, params=params, headers={"X-API-Key": USDA_API_KEY}, timeout=10)),
        ("header_bearer", lambda: requests.get(url, params=params, headers={"Authorization": f"Bearer {USDA_API_KEY}"}, timeout=10)),
        ("basic", lambda: requests.get(url, params=params, auth=(USDA_API_KEY, ""), timeout=10)),
    ]
    
    if debug:
        print(f"Trying USDA authentication methods...")
        print(f"API Key (first 20 chars): {USDA_API_KEY[:20]}...")
    
    for method_name, request_fn in methods:
        try:
            resp = request_fn()
            if debug:
                print(f"  {method_name}: {resp.status_code}")
            
            if resp.status_code == 200:
                _USDA_AUTH_METHOD = method_name
                print(f"✓ USDA auth method found: {method_name}")
                return resp
        except Exception as e:
            if debug:
                print(f"  {method_name}: ERROR - {e}")
    
    # If nothing worked, return the last response
    print(f"✗ All auth methods failed. Last response: {resp.status_code}")
    if debug and resp.status_code != 200:
        print(f"  Response text: {resp.text[:300]}")
    return resp

---
## 1. USDA Market News API — Illinois Soybean Cash Bids

The biggest unknown. We need: commodity cash bid prices by location in Illinois.

Base URL: `https://marsapi.ams.usda.gov/services/v1.2/`

Key endpoints to try:
- `/reports` — discover available reports
- Grain reports: look for `GX_GR110` (Illinois Grain) or similar

In [20]:
USDA_BASE = "https://marsapi.ams.usda.gov/services/v1.2"

# ── Step 1: discover grain-related reports ────────────────────────────────────
print("Testing USDA API authentication...\n")
resp = usda_request(
    f"{USDA_BASE}/reports",
    params={"q": "grain", "allSections": "true"},
    debug=True,
)
print(f"\nFinal Status: {resp.status_code}")

if resp.status_code == 200:
    reports = resp.json()
    if isinstance(reports, list):
        print(f"Total reports returned: {len(reports)}")
        df_reports = pd.DataFrame(reports)
        print(df_reports.columns.tolist())
        df_reports.head(20)
    else:
        print(f"Response type: {type(reports)}")
        pprint(reports if isinstance(reports, dict) else reports[:3])
else:
    print(f"Response body:\n{resp.text}")

Testing USDA API authentication...



UnboundLocalError: cannot access local variable 'resp' where it is not associated with a value

In [18]:
# ── Step 2: filter for Illinois grain / soybean reports ──────────────────────
il_reports = df_reports[
    df_reports.apply(
        lambda row: any(
            kw in str(row.values).lower()
            for kw in ["illinois", "soybean", "grain", "cash", "bid"]
        ),
        axis=1,
    )
]
print(f"Potentially relevant reports: {len(il_reports)}")
il_reports

NameError: name 'df_reports' is not defined

In [ ]:
# ── Step 3: pull a specific grain report ─────────────────────────────────────
CANDIDATE_SLUGS = [
    "GX_GR110",   # IL Grain — cash prices at country elevators
    "GX_GR111",
    "GX_GR120",
    "GX_GR130",
    "GX_GR210",
]

for slug in CANDIDATE_SLUGS:
    r = usda_request(f"{USDA_BASE}/reports/{slug}")
    print(f"{slug}: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"  Keys: {list(data.keys()) if isinstance(data, dict) else type(data)}")
        break

In [ ]:
# ── Step 4: try the report details endpoint ──────────────────────────────────
WORKING_SLUG = "GX_GR110"  # update if needed

r = usda_request(
    f"{USDA_BASE}/reports/{WORKING_SLUG}",
    params={"q": "soybean", "allSections": "true", "reportDate": TODAY},
)
print(f"Status: {r.status_code}")
if r.status_code == 200:
    data = r.json()
    pprint(data[:3] if isinstance(data, list) else data)
else:
    print(f"Error: {r.text[:300]}")

In [ ]:
# ── Step 5: alternative — search by commodity directly ────────────────────────
r = usda_request(
    f"{USDA_BASE}/reports",
    params={"q": "soybean Illinois cash", "allSections": "true"},
)
print(f"Status: {r.status_code}")
if r.status_code == 200:
    soy_reports = r.json()
    if isinstance(soy_reports, list):
        df_soy = pd.DataFrame(soy_reports)
        print(df_soy.shape)
        df_soy.head(10)
    else:
        print(f"Response: {soy_reports}")
else:
    print(f"Error: {r.text[:300]}")

---
## 2. yfinance — CBOT Soybean Futures (ZS=F)

We need: current front-month price, recent price history for trend chart, and ideally the forward curve.

In [21]:
# ── Front-month soybean futures ───────────────────────────────────────────────
ticker = yf.Ticker("ZS=F")

# Current price info
info = ticker.info
print("=== Ticker Info Keys ===")
print(list(info.keys()))
print()

# The fields we actually care about
fields = ["regularMarketPrice", "regularMarketPreviousClose", "bid", "ask",
          "fiftyTwoWeekHigh", "fiftyTwoWeekLow", "currency", "shortName",
          "contractSymbol", "expireDate"]
for f in fields:
    print(f"{f}: {info.get(f, 'N/A')}")

=== Ticker Info Keys ===
['maxAge', 'priceHint', 'previousClose', 'open', 'dayLow', 'dayHigh', 'regularMarketPreviousClose', 'regularMarketOpen', 'regularMarketDayLow', 'regularMarketDayHigh', 'volume', 'regularMarketVolume', 'averageVolume', 'averageVolume10days', 'averageDailyVolume10Day', 'bid', 'ask', 'bidSize', 'askSize', 'expireDate', 'openInterest', 'fiftyTwoWeekLow', 'fiftyTwoWeekHigh', 'allTimeHigh', 'allTimeLow', 'fiftyDayAverage', 'twoHundredDayAverage', 'currency', 'tradeable', 'quoteType', 'symbol', 'language', 'region', 'typeDisp', 'quoteSourceName', 'triggerable', 'customPriceAlertConfidence', 'contractSymbol', 'headSymbolAsString', 'marketState', 'corporateActions', 'regularMarketTime', 'underlyingSymbol', 'underlyingExchangeSymbol', 'exchange', 'exchangeTimezoneName', 'exchangeTimezoneShortName', 'gmtOffSetMilliseconds', 'market', 'esgPopulated', 'regularMarketPrice', 'fiftyDayAverageChange', 'fiftyDayAverageChangePercent', 'twoHundredDayAverageChange', 'twoHundredDayA

In [22]:
# ── 30-day price history ──────────────────────────────────────────────────────
hist = ticker.history(period="1mo", interval="1d")
print(f"Shape: {hist.shape}")
print(f"Columns: {hist.columns.tolist()}")
print()

# CBOT quotes soybeans in cents/bushel — confirm
print("Last 5 rows:")
print(hist.tail())

# Note: if price is ~1100-1200, it's in cents/bu. Divide by 100 for $/bu.
if hist["Close"].mean() > 100:
    print(f"\nLooks like cents/bu. Current: ${hist['Close'].iloc[-1]/100:.2f}/bu")
else:
    print(f"\nCurrent: ${hist['Close'].iloc[-1]:.2f}/bu")

Shape: (21, 7)
Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']

Last 5 rows:
                              Open     High      Low    Close  Volume  \
Date                                                                    
2026-05-04 00:00:00-04:00  1192.75  1210.75  1192.75  1207.50     845   
2026-05-05 00:00:00-04:00  1215.25  1215.25  1194.50  1195.75     540   
2026-05-06 00:00:00-04:00  1196.25  1196.25  1175.00  1179.00     252   
2026-05-07 00:00:00-04:00  1176.00  1178.50  1167.50  1177.00     276   
2026-05-08 00:00:00-04:00  1195.25  1210.25  1189.50  1208.00  101096   

                           Dividends  Stock Splits  
Date                                                
2026-05-04 00:00:00-04:00        0.0           0.0  
2026-05-05 00:00:00-04:00        0.0           0.0  
2026-05-06 00:00:00-04:00        0.0           0.0  
2026-05-07 00:00:00-04:00        0.0           0.0  
2026-05-08 00:00:00-04:00        0.0           0.0  

Looks 

In [23]:
# ── Forward curve: check nearby contract months ───────────────────────────────
# Soybean contract months: F(Jan) H(Mar) K(May) N(Jul) Q(Aug) U(Sep) X(Nov)
# Front-month ZS=F, next few: ZSN25, ZSQ25, ZSU25, ZSX25 etc.
# yfinance naming: ZSXXXXX.CBT or use the =F suffix for front month only

contract_tickers = ["ZS=F", "ZSN25.CBT", "ZSQ25.CBT", "ZSU25.CBT", "ZSX25.CBT"]
for sym in contract_tickers:
    t = yf.Ticker(sym)
    h = t.history(period="2d", interval="1d")
    if not h.empty:
        price = h["Close"].iloc[-1]
        print(f"{sym}: {price:.2f} ({'cents/bu → $'+str(round(price/100,2))+'/bu' if price > 100 else '$/bu'})") 
    else:
        print(f"{sym}: no data")

$ZSN25.CBT: possibly delisted; no price data found  (period=2d)


ZS=F: 1208.00 (cents/bu → $12.08/bu)
ZSN25.CBT: no data


$ZSQ25.CBT: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")
$ZSU25.CBT: possibly delisted; no price data found  (period=2d) (Yahoo error = "No data found, symbol may be delisted")


ZSQ25.CBT: no data
ZSU25.CBT: no data


$ZSX25.CBT: possibly delisted; no price data found  (period=2d)


ZSX25.CBT: no data


In [24]:
# ── Corn futures too (ZC=F) — we'll want this for config swap later ───────────
corn = yf.Ticker("ZC=F")
corn_hist = corn.history(period="5d", interval="1d")
print("Corn (ZC=F):")
print(corn_hist.tail(3))

Corn (ZC=F):
                             Open    High     Low   Close  Volume  Dividends  \
Date                                                                           
2026-05-06 00:00:00-04:00  463.00  463.00  452.25  452.75    1311        0.0   
2026-05-07 00:00:00-04:00  453.00  453.25  446.75  452.75     730        0.0   
2026-05-08 00:00:00-04:00  466.75  473.00  465.25  471.25  142817        0.0   

                           Stock Splits  
Date                                     
2026-05-06 00:00:00-04:00           0.0  
2026-05-07 00:00:00-04:00           0.0  
2026-05-08 00:00:00-04:00           0.0  


---
## 3. NOAA Weather — Drought / Condition Signals

We need: a simple drought/stress indicator for Central Illinois. Options:
- NOAA Climate Data Online (CDO) API — historical observations
- NOAA Drought Monitor feed (droughtmonitor.unl.edu) — shapefile or JSON
- NWS point forecast API (api.weather.gov) — current conditions at a lat/lon

Testing NWS point forecast first (no key needed).

In [25]:
# ── NWS Point Forecast — Springfield IL (lat 39.8, lon -89.6) ────────────────
# Step 1: get grid point
LAT, LON = 39.7817, -89.6501  # Springfield IL

r = requests.get(
    f"https://api.weather.gov/points/{LAT},{LON}",
    headers={"User-Agent": "Silo/0.1 (ehgarver@uchicago.edu)"},
    timeout=10,
)
print(f"Status: {r.status_code}")
point_data = r.json()
props = point_data.get("properties", {})
print(f"Grid: {props.get('gridId')} {props.get('gridX')},{props.get('gridY')}")
print(f"Forecast URL: {props.get('forecast')}")
print(f"Hourly URL: {props.get('forecastHourly')}")

Status: 200
Grid: ILX 47,55
Forecast URL: https://api.weather.gov/gridpoints/ILX/47,55/forecast
Hourly URL: https://api.weather.gov/gridpoints/ILX/47,55/forecast/hourly


In [26]:
# ── Step 2: pull the 7-day forecast ──────────────────────────────────────────
forecast_url = props.get("forecast")
r2 = requests.get(
    forecast_url,
    headers={"User-Agent": "Silo/0.1 (ehgarver@uchicago.edu)"},
    timeout=10,
)
print(f"Status: {r2.status_code}")
if r2.status_code == 200:
    forecast = r2.json()
    periods = forecast["properties"]["periods"]
    for p in periods[:5]:
        print(f"{p['name']}: {p['temperature']}°{p['temperatureUnit']}, {p['shortForecast']}, precip={p.get('probabilityOfPrecipitation', {}).get('value', 'N/A')}%")

Status: 200
Today: 69°F, Mostly Sunny, precip=0%
Tonight: 45°F, Mostly Clear, precip=0%
Monday: 74°F, Sunny, precip=0%
Monday Night: 48°F, Mostly Clear, precip=4%
Tuesday: 82°F, Mostly Sunny, precip=14%


In [27]:
# ── NOAA CDO API — recent precipitation / temperature observations ────────────
# No key needed for basic queries, but rate limited. Token optional.

# Springfield IL GHCND station: USW00014842 (Springfield Capital Airport)
r = requests.get(
    "https://www.ncdc.noaa.gov/cdo-web/api/v2/data",
    params={
        "datasetid": "GHCND",
        "stationid": "GHCND:USW00014842",
        "datatypeid": "PRCP,TMAX,TMIN",
        "startdate": THIRTY_DAYS_AGO,
        "enddate": TODAY,
        "limit": 100,
        "units": "standard",
    },
    headers={"token": NOAA_CDO_TOKEN} if NOAA_CDO_TOKEN else {},
    timeout=15,
)
print(f"CDO Status: {r.status_code}")
if r.status_code == 200:
    cdo_data = r.json()
    df_cdo = pd.DataFrame(cdo_data.get("results", []))
    print(df_cdo.head(10))
else:
    print(f"Error: {r.text[:300]}")

CDO Status: 400
Error: {"status" : "400", "message" : "Token parameter is required."}


In [28]:
# ── Drought Monitor — tabular JSON feed (no auth needed) ─────────────────────
# Weekly drought stats by state
r = requests.get(
    "https://usdm.climate.unl.edu/DmData/Severity/GetDmScoreDataDownload",
    params={
        "county": False,
        "state": "IL",
        "startdate": THIRTY_DAYS_AGO,
        "enddate": TODAY,
    },
    timeout=15,
)
print(f"Drought Monitor status: {r.status_code}")
if r.status_code == 200:
    print("Content type:", r.headers.get("content-type"))
    print(r.text[:500])
else:
    print(r.text[:300])

ConnectionError: HTTPSConnectionPool(host='usdm.climate.unl.edu', port=443): Max retries exceeded with url: /DmData/Severity/GetDmScoreDataDownload?county=False&state=IL&startdate=2026-04-10&enddate=2026-05-10 (Caused by NameResolutionError("HTTPSConnection(host='usdm.climate.unl.edu', port=443): Failed to resolve 'usdm.climate.unl.edu' ([Errno 8] nodename nor servname provided, or not known)"))

---
## 4. FRED — Diesel Prices & T-Bill Rate

We need:
- Diesel price (used in transport cost formula: $/bu/mile → convert from diesel $/gal)
- 3-month T-bill rate (risk-free rate for storage opportunity cost)

FRED series:
- `GASDESW` — U.S. weekly diesel retail price ($/gallon)
- `DTB3` — 3-month T-bill secondary market rate

In [29]:
try:
    from fredapi import Fred

    if FRED_API_KEY:
        fred = Fred(api_key=FRED_API_KEY)

        # ── Diesel price ──────────────────────────────────────────────────────────
        diesel = fred.get_series("GASDESW", observation_start=THIRTY_DAYS_AGO)
        print("Diesel ($/gal) — last 4 weeks:")
        print(diesel.tail(4))
        diesel_current = diesel.iloc[-1]
        print(f"\nCurrent: ${diesel_current:.3f}/gal")

        # ── T-bill rate ───────────────────────────────────────────────────────────
        tbill = fred.get_series("DTB3", observation_start=THIRTY_DAYS_AGO)
        print("\n3-Month T-Bill rate (%) — last week:")
        print(tbill.tail(5))
        tbill_current = tbill.dropna().iloc[-1]
        print(f"\nCurrent: {tbill_current:.2f}%")
    else:
        print("Set FRED_API_KEY to test this cell. Free key: https://fred.stlouisfed.org/docs/api/api_key.html")
except ImportError:
    print("fredapi not installed. Using HTTP fallback in next cell...")

fredapi not installed. Using HTTP fallback in next cell...


In [30]:
# ── FRED without a key — raw HTTP fallback ────────────────────────────────────
# FRED has a public API that works without a key for simple reads (rate limited)
def fred_series_no_key(series_id, start=THIRTY_DAYS_AGO):
    r = requests.get(
        "https://api.stlouisfed.org/fred/series/observations",
        params={
            "series_id": series_id,
            "observation_start": start,
            "file_type": "json",
            "api_key": FRED_API_KEY or "abcdefghijklmnopqrstuvwxyz123456",  # fallback if no key
        },
        timeout=10,
    )
    return r

r_diesel = fred_series_no_key("GASDESW")
print(f"Diesel fetch status: {r_diesel.status_code}")
if r_diesel.status_code == 200:
    obs = r_diesel.json().get("observations", [])
    df_diesel = pd.DataFrame(obs)
    df_diesel["value"] = pd.to_numeric(df_diesel["value"], errors="coerce")
    print(df_diesel.tail(4))

r_tbill = fred_series_no_key("DTB3")
print(f"\nT-bill fetch status: {r_tbill.status_code}")
if r_tbill.status_code == 200:
    obs = r_tbill.json().get("observations", [])
    df_tbill = pd.DataFrame(obs)
    df_tbill["value"] = pd.to_numeric(df_tbill["value"], errors="coerce")
    print(df_tbill.dropna().tail(4))

Diesel fetch status: 200
  realtime_start realtime_end        date  value
0     2026-05-10   2026-05-10  2026-04-13  5.608
1     2026-05-10   2026-05-10  2026-04-20  5.403
2     2026-05-10   2026-05-10  2026-04-27  5.351
3     2026-05-10   2026-05-10  2026-05-04  5.640

T-bill fetch status: 200
   realtime_start realtime_end        date  value
16     2026-05-10   2026-05-10  2026-05-04   3.61
17     2026-05-10   2026-05-10  2026-05-05   3.61
18     2026-05-10   2026-05-10  2026-05-06   3.61
19     2026-05-10   2026-05-10  2026-05-07   3.61


---
## 5. Google Maps Distance Matrix

Demo scenario: Springfield IL farm → Buyer A (local elevator) + Buyer B (~28 miles away)

We need: driving distance in miles for transport cost formula.

In [31]:
DEMO_FARM = "Springfield, IL 62701"          # farmer origin
DEMO_BUYER_A = "Williamsville, IL 62693"     # nearby elevator
DEMO_BUYER_B = "Petersburg, IL 62675"        # ~28 miles

if GOOGLE_MAPS_KEY:
    r = requests.get(
        "https://maps.googleapis.com/maps/api/distancematrix/json",
        params={
            "origins": DEMO_FARM,
            "destinations": "|".join([DEMO_BUYER_A, DEMO_BUYER_B]),
            "units": "imperial",
            "key": GOOGLE_MAPS_KEY,
        },
        timeout=10,
    )
    print(f"Status: {r.status_code}")
    data = r.json()
    print(f"Status field: {data.get('status')}")

    rows = data.get("rows", [{}])[0].get("elements", [])
    destinations = data.get("destination_addresses", [])
    for dest, el in zip(destinations, rows):
        if el.get("status") == "OK":
            miles = el["distance"]["value"] / 1609.34  # meters → miles
            print(f"  {dest}: {miles:.1f} miles, {el['duration']['text']}")
        else:
            print(f"  {dest}: {el.get('status')}")
else:
    print("Set GOOGLE_MAPS_API_KEY to test this cell.")
    print("Fallback: geopy straight-line distance (shown below)")

Status: 200
Status field: REQUEST_DENIED


IndexError: list index out of range

In [32]:
# ── Geopy fallback — no API key needed ───────────────────────────────────────
# Uses straight-line (haversine) distance — less accurate but zero cost
# Install if needed: pip install geopy
try:
    from geopy.distance import geodesic
    from geopy.geocoders import Nominatim

    geolocator = Nominatim(user_agent="silo_test")

    def geocode(address):
        loc = geolocator.geocode(address)
        return (loc.latitude, loc.longitude) if loc else None

    farm_coords = geocode(DEMO_FARM)
    buyer_a_coords = geocode(DEMO_BUYER_A)
    buyer_b_coords = geocode(DEMO_BUYER_B)

    print(f"Farm: {farm_coords}")
    print(f"Buyer A: {buyer_a_coords}")
    print(f"Buyer B: {buyer_b_coords}")

    if all([farm_coords, buyer_a_coords, buyer_b_coords]):
        dist_a = geodesic(farm_coords, buyer_a_coords).miles
        dist_b = geodesic(farm_coords, buyer_b_coords).miles
        print(f"\nBuyer A: {dist_a:.1f} miles (straight-line)")
        print(f"Buyer B: {dist_b:.1f} miles (straight-line)")
        print("Note: driving distance typically 15-30% longer than straight-line")
except ImportError:
    print("geopy not installed. Run: pip install geopy")

geopy not installed. Run: pip install geopy


---
## 6. Transport Net Revenue — Quick Sanity Check

Formula: `net_revenue = (bid × qty) − (distance_miles × qty × cost_per_bu_per_mile)`

Default cost: `$0.042/bu/mile` (rough industry figure — adjust with diesel price later)

In [ ]:
QTY = 42_000  # bushels
TRANSPORT_COST_PER_BU_MILE = 0.042  # $/bu/mile

scenarios = [
    {"name": "Buyer A (local elevator)",  "bid": 11.42, "distance_miles": 8},
    {"name": "Buyer B (Springfield Co-op)", "bid": 11.61, "distance_miles": 28},
]

results = []
for s in scenarios:
    gross = s["bid"] * QTY
    transport = s["distance_miles"] * QTY * TRANSPORT_COST_PER_BU_MILE
    net = gross - transport
    net_per_bu = net / QTY
    results.append({
        "Buyer": s["name"],
        "Bid ($/bu)": s["bid"],
        "Distance (mi)": s["distance_miles"],
        "Gross Revenue": f"${gross:,.0f}",
        "Transport Cost": f"${transport:,.0f}",
        "Net Revenue": f"${net:,.0f}",
        "Net $/bu": f"${net_per_bu:.3f}",
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print()

# Expected: Buyer B wins despite higher transport
nets = [r["Net Revenue"] for r in results]
print("Winner:", results[0]["Buyer"] if results[0]["Net Revenue"] > results[1]["Net Revenue"] else results[1]["Buyer"])

---
## 7. Seasonal Tendency — Historical Basis Pattern

We need 5-year weekly average price changes for IL soybeans to power the "wait" scenario.
Using yfinance to pull historical ZS futures data and compute week-of-year averages.

In [ ]:
# ── Pull 5 years of ZS futures daily closes ───────────────────────────────────
zs = yf.Ticker("ZS=F")
hist_5y = zs.history(period="5y", interval="1wk")
print(f"Shape: {hist_5y.shape}")
print(hist_5y.head())

In [ ]:
# ── Compute week-over-week % change, group by week-of-year ───────────────────
hist_5y = hist_5y.copy()
hist_5y["close_cents"] = hist_5y["Close"]
hist_5y["close_dollars"] = hist_5y["Close"] / 100  # cents → dollars
hist_5y["pct_change"] = hist_5y["Close"].pct_change() * 100
hist_5y["week_of_year"] = hist_5y.index.isocalendar().week.astype(int)

seasonal = (
    hist_5y.groupby("week_of_year")["pct_change"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "avg_pct_change", "std": "std_pct_change", "count": "n_obs"})
)

current_week = date.today().isocalendar().week
print(f"Current week of year: {current_week}")
print()
print("Seasonal tendency — next 8 weeks:")
print(seasonal.loc[current_week : current_week + 7].round(3))

In [ ]:
# ── 2-week and 4-week expected value from seasonal data ───────────────────────
current_price = hist_5y["close_dollars"].iloc[-1]

def compound_seasonal_return(start_week, n_weeks):
    """Compound average weekly pct changes over n_weeks from start_week."""
    weeks = [(start_week + i - 1) % 52 + 1 for i in range(n_weeks)]
    changes = [seasonal.loc[w, "avg_pct_change"] / 100 for w in weeks if w in seasonal.index]
    return np.prod([1 + c for c in changes]) - 1

ret_2w = compound_seasonal_return(current_week, 2)
ret_4w = compound_seasonal_return(current_week, 4)

print(f"Current futures price: ${current_price:.2f}/bu")
print(f"Expected price in 2 weeks (seasonal avg): ${current_price * (1 + ret_2w):.2f}/bu  ({ret_2w*100:+.2f}%)")
print(f"Expected price in 4 weeks (seasonal avg): ${current_price * (1 + ret_4w):.2f}/bu  ({ret_4w*100:+.2f}%)")

---
## 8. API Health Summary

Run this cell last to get a quick status board.

In [ ]:
import sys

def check(label, fn):
    try:
        result = fn()
        print(f"  ✓  {label}: {result}")
    except Exception as e:
        print(f"  ✗  {label}: {e}")

print("=" * 60)
print("API Health Check")
print("=" * 60)

# yfinance
check(
    "yfinance ZS=F",
    lambda: f"${yf.Ticker('ZS=F').history(period='2d')['Close'].iloc[-1]/100:.2f}/bu",
)

# USDA
def usda_check():
    r = usda_request(f"{USDA_BASE}/reports", params={"q": "grain"})
    r.raise_for_status()
    return f"{len(r.json())} reports found"
check("USDA Market News (with API key)", usda_check)

# NWS
def nws_check():
    r = requests.get(
        f"https://api.weather.gov/points/{LAT},{LON}",
        headers={"User-Agent": "Silo/0.1"},
        timeout=8,
    )
    r.raise_for_status()
    return "OK — point data returned"
check("NOAA NWS", nws_check)

# FRED
def fred_check():
    r = fred_series_no_key("GASDESW")
    r.raise_for_status()
    obs = r.json().get("observations", [])
    if not obs:
        raise ValueError("empty observations")
    latest = next((o for o in reversed(obs) if o["value"] != "."), None)
    return f"diesel ${float(latest['value']):.3f}/gal on {latest['date']}"
check("FRED diesel (GASDESW)", fred_check)

# Google Maps
def gmaps_check():
    if not GOOGLE_MAPS_KEY:
        return "SKIPPED — no key set"
    r = requests.get(
        "https://maps.googleapis.com/maps/api/distancematrix/json",
        params={"origins": DEMO_FARM, "destinations": DEMO_BUYER_B, "key": GOOGLE_MAPS_KEY},
        timeout=8,
    )
    r.raise_for_status()
    status = r.json().get("status")
    if status != "OK":
        raise ValueError(status)
    el = r.json()["rows"][0]["elements"][0]
    return f"{el['distance']['text']}, {el['duration']['text']}"
check("Google Maps Distance Matrix", gmaps_check)

print("=" * 60)